### 1. Initialization and read


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, date_sub, lead
from pyspark.sql.window import Window

In [0]:
df = spark.table("catalogue_project1.bronze.crm_prd_info")

### 2. Transformations
2.1 Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

2.2 Parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), '-', '_'))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

2.3 Dealing with zeros and NULLs

In [0]:
df = df.withColumn('prd_cost', F.coalesce(col("prd_cost"), F.lit(0)))

2.4 Standarization

In [0]:
df = (
    df.withColumn(
        "prd_line",
        F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
         .when(F.upper(F.col("prd_line")) == "R", "Road")
         .when(F.upper(F.col("prd_line")) == "S", "Other Sales")
         .when(F.upper(F.col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

2.5 Date casting

In [0]:
df = df.withColumn("prd_start_date", col("prd_start_dt").cast(DateType()))

# TO DO: find a way to deal with NULLs in prd_end_dt. 

2.6 Rename columns

In [0]:
RENAME_CONFIG = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}
for old_name, new_name in RENAME_CONFIG.items():
    df = df.withColumnRenamed(old_name, new_name)

* Intermediate step: check the changes applied correctly.

In [0]:
df.limit(10).display()

### 3. Write into the Silver delta table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("catalogue_project1.silver.crm_products")

* Check that everything went well

In [0]:
%sql
SELECT * FROM catalogue_project1.silver.crm_products LIMIT 10